# Backtest a trading strategy on `ig_market_data.db`

Loads historical candles for one `epic` / `resolution` from
`data/ig_market_data.db` (produced by `02_ig_import_csv_to_db.ipynb` from a CSV
`01_ig_download_prices.ipynb` downloaded), restricts them to a chosen date window, runs a
strategy through the small backtester in `ig_backtest.py`, and shows the
results: a metrics table (**initial funding -> final funding**), the
round-trip trade list, and price / equity / drawdown charts.

**Engine (`ig_backtest.py`)** - a CFD-style fixed-size position: the account
starts with `INITIAL_FUNDING` (default **20,000 AUD**), one index point is
worth `POINT_VALUE` per contract (default **1.0 AUD/point**), and `SIZE`
contracts are held, long or short. Signals are BUY / SELL / HOLD, traded the
way `ig_quant_trading` trades them live - an opposite signal closes the
position and opens the reverse one, HOLD keeps it - with fills at
the **next bar's open** after a signal (no look-ahead), a flat `fee_bps` cost
per entry and exit, prices are the bid/ask mid. Teaching-grade, not an
execution simulator.

**Strategy** - the default is `SMAStrategy`: BUY (long) while the fast simple
moving average of the close is above the slow one, SELL (short) while it's below.
`BuyHoldStrategy` is included as the benchmark. Strategies come from
`ig_quant_trading.strategies` (imported below as `st`) - the sibling
ig-quant-trading repo that trades them live, installed here editable, so
what you backtest is exactly what it runs - every strategy it has (MACD,
Stochastic, ADX, CCI, ...) can be dropped in here. Add your own there by
subclassing `VectorizedStrategy` and implementing `signals(frame)`.

**Prerequisite:** the `trading_toolkit` package installed —
`uv sync --extra dev` from the repo root (see the README); brings in
`python-dotenv`, `pandas`, `matplotlib`.

## 1. Config, window & strategy

`EPIC` / `RESOLUTION` / `DB_PATH` come from `ig_config.py` (`.env`), same as
the other notebooks. Set `START` / `END` to bound the backtest (either may be
`None`), and pick the strategy + its parameters here.

In [ ]:
import sqlite3

import pandas as pd

from ig_quant_trading import strategies as st

from trading_toolkit import ig_backtest as backtest
from trading_toolkit.ig_candle_db import load_candles
from trading_toolkit.ig_config import Config
from trading_toolkit.constants.ig_ohlc_fields import OHLCField

cfg = Config.from_env()
EPIC = cfg.epic
RESOLUTION = cfg.resolution
DB_PATH = cfg.db_path

# Backtest window (inclusive). None = use all available history on that side.
START = "2021-01-01"
END = None

# Strategy under test.
strategy = st.SMAStrategy(fast_period=10, slow_period=30)

# Account model.
INITIAL_FUNDING = 20_000.0  # starting balance, account currency
POINT_VALUE = 1.0           # account-currency P&L per index point, per contract
SIZE = 1.0                  # contracts held, long or short
CURRENCY = "AUD"
FEE_BPS = 1.0               # cost per entry and per exit, basis points of notional

print(f"{EPIC}  {RESOLUTION}   DB: {DB_PATH}")
print(f"window  : {START or 'earliest'}  ->  {END or 'latest'}")
print(f"strategy: {strategy.label}")
print(f"account : {INITIAL_FUNDING:,.0f} {CURRENCY}   {SIZE:g} contract(s) @ {POINT_VALUE:g} {CURRENCY}/point   fee {FEE_BPS} bps")

## 2. Load candles from `ig_market_data.db`

`ig_candle_db.load_candles` yields one flat OHLC dict per candle (mid prices);
here they become a `DatetimeIndex`ed DataFrame clipped to the window.

In [ ]:
conn = sqlite3.connect(DB_PATH)
df = pd.DataFrame(load_candles(conn, RESOLUTION, epic=EPIC))
conn.close()

assert not df.empty, (
    f"No candles for {EPIC} / {RESOLUTION} in {DB_PATH} - run 01_ig_download_prices.ipynb "
    "then 02_ig_import_csv_to_db.ipynb first."
)

df[OHLCField.SNAPSHOT_TIME_UTC] = pd.to_datetime(
    df[OHLCField.SNAPSHOT_TIME_UTC], format="%Y-%m-%dT%H:%M:%S"
)
df = df.set_index(OHLCField.SNAPSHOT_TIME_UTC).sort_index()
df = df[[OHLCField.OPEN, OHLCField.HIGH, OHLCField.LOW, OHLCField.CLOSE, OHLCField.VOLUME]]

if START:
    df = df[df.index >= pd.Timestamp(START)]
if END:
    df = df[df.index <= pd.Timestamp(END)]

assert len(df) > 50, (
    f"Only {len(df)} bars in the window - widen START/END or download more history."
)
print(f"{len(df)} bars   {df.index[0]:%Y-%m-%d %H:%M}  ->  {df.index[-1]:%Y-%m-%d %H:%M}")
df.head()

## 3. Run the backtest

`run_backtest` returns a `BacktestResult` (equity curve, trades, stats).
`.summary()` prints the headline metrics; buy & hold over the same window is
shown for comparison.

In [ ]:
result = backtest.run_backtest(
    df, strategy,
    initial_funding=INITIAL_FUNDING, point_value=POINT_VALUE, size=SIZE,
    fee_bps=FEE_BPS, currency=CURRENCY,
)
print(result.summary())

## 4. Trades

One row per round-trip (entry -> exit). `direction` is BUY (long) or SELL
(short); `points` is the price move in the trade's favour,
`pnl` is that move in `CURRENCY` (`points * POINT_VALUE * SIZE`), `open =
True` marks a position still held at the end of the window, marked out at the
last mid for accounting.

In [ ]:
trades = result.trades.copy()
if not trades.empty:
    trades["return_pct"] = trades["return_pct"].round(2)
trades

## 5. Charts

Price with the moving averages and buy/sell markers, the strategy equity
curve against buy & hold, and the strategy drawdown.

In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt

close = df[OHLCField.CLOSE]
eq = result.equity
# buy & hold account value: same funding, same SIZE contracts held throughout
bh = INITIAL_FUNDING + (close - close.iloc[0]) * POINT_VALUE * SIZE
dd = (eq / eq.cummax() - 1) * 100
t = result.trades

fig, (ax_px, ax_eq, ax_dd) = plt.subplots(
    3, 1, sharex=True, figsize=(13, 10),
    gridspec_kw={"height_ratios": [3, 2, 1]},
)

ax_px.plot(close.index, close, color="#455a64", lw=1, label="close (mid)")
if hasattr(strategy, "fast_period"):
    ax_px.plot(close.index, close.rolling(strategy.fast_period).mean(), color="#26a69a", lw=1,
               label=f"SMA {strategy.fast_period}")
    ax_px.plot(close.index, close.rolling(strategy.slow_period).mean(), color="#ef5350", lw=1,
               label=f"SMA {strategy.slow_period}")
if not t.empty:
    for side, marker, colour, label in (("BUY", "^", "#2e7d32", "open long"),
                                        ("SELL", "v", "#c62828", "open short")):
        e = t[t["direction"] == side]
        ax_px.scatter(e["entry_time"], e["entry_price"], marker=marker, s=70, color=colour,
                      zorder=5, label=label)
    closed = t[~t["open"]]
    ax_px.scatter(closed["exit_time"], closed["exit_price"], marker="x", s=46, color="#212121",
                  zorder=6, label="close")
ax_px.set_ylabel("price (mid)")
ax_px.set_title(f"{EPIC}  {RESOLUTION}  -  {strategy.label}")
ax_px.legend(loc="upper left")
ax_px.grid(alpha=0.3)

ax_eq.axhline(INITIAL_FUNDING, color="#bbbbbb", lw=0.8)
ax_eq.plot(eq.index, eq, color="#1e88e5", lw=1.3, label="strategy")
ax_eq.plot(bh.index, bh, color="#9e9e9e", lw=1, ls="--", label="buy & hold")
ax_eq.set_ylabel(f"funding ({CURRENCY})")
ax_eq.legend(loc="upper left")
ax_eq.grid(alpha=0.3)

ax_dd.fill_between(dd.index, dd, 0, color="#ef5350", alpha=0.4)
ax_dd.set_ylabel("drawdown %")
ax_dd.grid(alpha=0.3)

ax_dd.xaxis.set_major_locator(mdates.AutoDateLocator())
ax_dd.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 6. Compare a few strategies

Same window, same costs - a quick sweep so the numbers above have something
to sit against. Edit the list to taste.

In [ ]:
candidates = [
    st.BuyHoldStrategy(),
    st.SMAStrategy(5, 20),
    st.SMAStrategy(10, 30),
    st.SMAStrategy(20, 50),
    st.SMAStrategy(50, 100),
]

rows = []
for strat in candidates:
    try:
        s = backtest.run_backtest(
            df, strat,
            initial_funding=INITIAL_FUNDING, point_value=POINT_VALUE, size=SIZE,
            fee_bps=FEE_BPS, currency=CURRENCY,
        ).stats
    except ValueError as exc:  # e.g. slow window longer than the data
        print(f"skip {strat.label}: {exc}")
        continue
    rows.append({
        "strategy": strat.label,
        f"final_funding_{CURRENCY}": round(s["final_funding"], 2),
        f"profit_{CURRENCY}": round(s["profit"], 2),
        "return_%": round(s["total_return_pct"], 2),
        "max_dd_%": round(s["max_drawdown_pct"], 2),
        "sharpe": round(s["sharpe"], 2),
        "exposure_%": round(s["exposure_pct"], 1),
        "trades": s["n_trades"],
    })

pd.DataFrame(rows).set_index("strategy")